In [ ]:
#1. Groq setup in Google Colab
#Cell 1 — Install libraries

In [ ]:
!pip install -q groq fastmcp gradio requests pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.


In [ ]:
# Retrieve the Groq API key from Colab User Data (Secrets) and set it as an environment variable
import os
from google.colab import userdata

try:
    # This expects you to have added a Secret named 'GROQ_API_KEY' in the Colab Left Sidebar (key icon)
    os.environ["GROQ_API_KEY"] = userdata.get("jnnce_fdp")
    print("Groq API Key successfully configured from Colab secrets!")
except userdata.SecretNotFoundError:
    print("Error: 'GROQ_API_KEY' secret not found. Please click on the key icon in the left sidebar and add a secret named 'GROQ_API_KEY'.")

Groq API Key successfully configured from Colab secrets!


In [ ]:
from groq import Groq
import os

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

# Switch to a supported Groq model with higher rate limits
MODEL = "openai/gpt-oss-120b"

In [ ]:
def ask_groq(prompt):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are an intelligent AI research assistant.
                Answer clearly, accurately and in a structured format.
                """
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content


ask_groq("Explain Agentic AI")

'**Agentic AI – A Structured Overview**\n\n---\n\n## 1. What Is Agentic AI?\n\n| Aspect | Description |\n|--------|-------------|\n| **Definition** | An *agentic* AI system is an artificial intelligence that exhibits **autonomous decision‑making**, **goal‑directed behavior**, and the ability to **act in an environment** to achieve its objectives, often with minimal human intervention. |\n| **Key Characteristics** | - **Agency**: Possesses a sense of “self‑directedness” (i.e., it can set sub‑goals, plan, and execute actions). <br> - **Perception–Action Loop**: Continuously senses the environment, processes information, and takes actions. <br> - **Adaptivity**: Learns from feedback and updates its policies. <br> - **Persistence**: Maintains internal state across time steps. |\n| **Contrast with Traditional AI** | Traditional AI (e.g., classification models, static recommendation engines) typically **react** to inputs without a persistent goal or long‑term planning. Agentic AI **acts** an

In [ ]:
def calculate_growth(current, previous):

    if previous == 0:
        return 0

    return round(
        ((current - previous) / previous) * 100,
        2
    )

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_growth",
            "description": "Calculate percentage growth between two values",
            "parameters": {
                "type": "object",
                "properties": {
                    "current": {
                        "type": "number"
                    },
                    "previous": {
                        "type": "number"
                    }
                },
                "required": ["current", "previous"]
            }
        }
    }
]

In [ ]:
import json

available_functions = {
    "calculate_growth": calculate_growth
}


def execute_tool(tool_call):

    function_name = tool_call.function.name

    arguments = json.loads(
        tool_call.function.arguments
    )

    function = available_functions[function_name]

    result = function(**arguments)

    return str(result)

In [ ]:
def groq_agent_with_tools(user_prompt):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI agent. Use tools when needed."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    messages.append(message)

    if message.tool_calls:

        for tool_call in message.tool_calls:

            result = execute_tool(tool_call)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": result
            })

        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return final_response.choices[0].message.content

    return message.content

In [ ]:
groq_agent_with_tools(
    "Revenue increased from 100 crore to 125 crore. "
    "Calculate the growth and explain it."
)

'**Growth Calculation**\n\n- **Previous revenue:**\u202f100\u202fcrore  \n- **Current revenue:**\u202f125\u202fcrore  \n\n**Growth amount:**\u202f125\u202f–\u202f100\u202f=\u202f**25\u202fcrore**  \n\n**Growth rate:**\u202f\\( \\frac{25}{100} \\times 100\\% = \\) **25\u202f%**\n\n**Explanation**\n\nThe revenue grew by 25\u202fcrore, which represents a 25\u202f% increase over the original 100\u202fcrore. This indicates a healthy expansion—if this trend continues, the business is on track for continued growth.'

In [ ]:
def research_agent(topic):

    prompt = f"""
    You are a Research Agent.

    Research the following topic:
    {topic}

    Provide:
    - Background
    - Current trends
    - Important developments
    - Opportunities
    - Challenges
    """

    return ask_groq(prompt)

In [ ]:
def analyst_agent(research):

    prompt = f"""
    You are a Data Analyst Agent.

    Analyze the following research:

    {research}

    Provide:
    - Key insights
    - Opportunities
    - Risks
    - SWOT analysis
    - Strategic recommendations
    """

    return ask_groq(prompt)

In [ ]:
def report_agent(topic, research, analysis):
    # Aggressively reduce characters to avoid exceeding the strict 8000 TPM limit
    short_research = research[:500] if len(research) > 500 else research
    short_analysis = analysis[:500] if len(analysis) > 500 else analysis

    prompt = f"""
    Write a concise summary report on '{topic}'.

    Research:
    {short_research}

    Analysis:
    {short_analysis}

    Format: Executive Summary | Insights | Recommendations.
    """

    return ask_groq(prompt)

In [ ]:
def orchestrator(topic):

    print(" Starting AI Research Workflow")

    print("\n Calling Research Agent...")

    research = research_agent(topic)

    print("\n Calling Data Analyst Agent...")

    analysis = analyst_agent(research)

    print("\n Calling Report Agent...")

    report = report_agent(
        topic,
        research,
        analysis
    )

    return {
        "topic": topic,
        "research": research,
        "analysis": analysis,
        "report": report
    }

In [ ]:
# Re-running the orchestration workflow using our updated MODEL
result = orchestrator(
    "Agentic AI opportunities in India"
)

print(result["report"])

 Starting AI Research Workflow

 Calling Research Agent...

 Calling Data Analyst Agent...

 Calling Report Agent...
**Agentic AI – Opportunities in India (Nov 2026)**  
*Concise Summary Report*  

---

## 1. Executive Summary  
Agentic AI—autonomous systems that perceive, reason, plan and act—has moved from a research curiosity to a commercial catalyst in India. A confluence of a large, English‑proficient talent pool, rapid cloud adoption, supportive policy (National AI Strategy 2025‑30), and a burgeoning startup ecosystem is creating a fertile ground for agentic AI across three core verticals: **Enterprise productivity, Consumer‑facing services, and Industry‑4.0 automation**. While the market is projected to reach **≈ US$ 12 bn by 2032**, success hinges on navigating data‑privacy regulations, talent scarcity in advanced RL/ML‑Ops, and the need for robust governance frameworks.

---

## 2. Market Landscape  

| Dimension | Current State (2026) | Near‑Term Outlook (2027‑30) |
|--------

In [1]:
import gradio as gr


def run_workflow(topic):

    result = orchestrator(topic)

    return (
        result["research"],
        result["analysis"],
        result["report"]
    )


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="PragyanAI -Agentic AI Research System"
) as demo:

    gr.Markdown("""
    # Agentic AI Research Assistant

    ### Groq LLM + A2UI + MCP + A2A
    """)

    topic = gr.Textbox(
        label="Enter Research Topic",
        placeholder="e.g. Agentic AI in Healthcare"
    )

    run_button = gr.Button(
        "Run Agentic AI Workflow",
        variant="primary"
    )

    with gr.Tab(" Research"):

        research_output = gr.Markdown()

    with gr.Tab(" Analysis"):

        analysis_output = gr.Markdown()

    with gr.Tab(" Final Report"):

        report_output = gr.Markdown()

    run_button.click(
        fn=run_workflow,
        inputs=topic,
        outputs=[
            research_output,
            analysis_output,
            report_output
        ]
    )


demo.launch(share=True)

/tmp/ipykernel_889/4028393603.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2910e00d5bcd125e99.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
